In [ ]:
import pandas as pd

# Paths
mapping_file = "D:/Tushar/main_with_subs_only.xlsx"
indent_file  = "D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

# Load
df_mapping = pd.read_excel(mapping_file)
df_indent   = pd.read_excel(indent_file)

# ── DIAGNOSTICS ──
print("Columns in main_with_subs_only.xlsx:")
print(df_mapping.columns.tolist())

print("\nColumns in Monthly Indent.xlsx:")
print(df_indent.columns.tolist())

# Try to find the part number column automatically (common names)
possible_part_cols = ['Part number', 'Part Number', 'Part No.', 'Part No', 'PART NUMBER', 'Part#', 'Item', 'Code']
part_col = next((col for col in possible_part_cols if col in df_indent.columns), None)

if part_col:
    print(f"\nFound part number column: '{part_col}' → renaming to 'Switch_Part'")
    df_indent = df_indent.rename(columns={part_col: 'Switch_Part'})
else:
    print("\nERROR: Could not find part number column. Please check the print above and rename manually.")
    # You can stop here or raise error
    raise ValueError("Part number column not found - check printed columns")

# Rename mapping columns (should already be ok)
df_mapping = df_mapping.rename(columns={
    'Main_Label': 'Child_Part',
    'Sub_Label':  'Switch_Part',
    'Main_Count': 'Qty_per_Switch',
    'Sub_Count':  'Historical_Total'
})

# Month columns – use the actual columns after rename
month_cols = [col for col in df_indent.columns if "'" in col and col != 'Switch_Part']

print("\nDetected month columns:", month_cols)
if not month_cols:
    print("WARNING: No month columns with ' detected. Check file structure.")

clean_months = [m.replace("'", "") for m in month_cols]

# Merge
df_merged = pd.merge(
    df_mapping[['Child_Part', 'Switch_Part', 'Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'
)

# Rest of your calculation code...
for month, clean in zip(month_cols, clean_months):
    daily_col   = f"Daily_{clean}"
    twodays_col = f"2Days_{clean}"
    df_merged[daily_col]   = (df_merged[month] / 30.0).round(2)
    df_merged[twodays_col] = (df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2).round(2)

# Totals
agg_dict = {f"Daily_{clean}": 'sum' for clean in clean_months}
totals = df_merged.groupby('Child_Part', as_index=False).agg(agg_dict)

for clean in clean_months:
    totals[f"2Days_{clean}"] = (totals[f"Daily_{clean}"] * 2).round(2)

totals_cols = (
    ['Child_Part'] +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
)
totals = totals[totals_cols]

totals.to_excel("Child_Totals_2Days_Per_Month.xlsx", index=False)
print(f"Totals saved ({len(totals)} rows)")

# Detailed
detailed_cols = (
    ['Child_Part', 'Switch_Part', 'Qty_per_Switch'] +
    month_cols +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
)
detailed = df_merged[detailed_cols]
detailed.to_excel("Child_Detailed_Breakdown_2Days.xlsx", index=False)
print(f"Detailed saved ({len(detailed)} rows)")

print("Done!")